In [ ]:
from pathlib import Path
import xarray as xr
import utils
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import umap
from sklearn.datasets import load_digits
import re
import json
import pickle
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as CFeature
import random
from scipy.stats import bootstrap


## Start in code cell 7 if CombinedDataset.nc is already created.


Some Additional Filtering that wasn't included in the original data consolidation workflow

In [ ]:
# The path for HAFS consolidated Date
folder_path = 'Model_Data/CompleteData/Jul9Run/Consolidated'

# HAFS Data is in multiple NC files. Consolidate_Data creates one dataset
ds = utils.consolidate_data(folder_path)


Function to get frame intensity change. Lead time is customizable but defaults to 24 hours in the future.

In [ ]:
# Checking for NA's
ds_cleaned = ds.dropna(dim = 'frame_number', how = 'any')
# Limiting frames to tropical storm strength or stronger (17 m/s or stronger)
ds_cleaned_1 = ds_cleaned.where(ds_cleaned['max_wind'] >= 17, drop = True)

# Only using storms in the Northern Hemisphere
ds_final = ds_cleaned.where(ds_cleaned['center_lat'] > 0, drop = True)
# ds_cleaned = ds_cleaned.dropna(dim = 'frame_number', how = 'any')



# with open('intensity_change.pkl', 'rb') as file:
#     dict = pickle.load(file)

In [ ]:
ds_cleaned = ds.dropna(dim = 'frame_number', how = 'any')

In [ ]:
ds_cleaned.to_netcdf('Semester2_TC_Fields.nc',
                     engine='h5netcdf')

# for name, var in ds.variables.items():
#     print(f"Variable: {name} | dtype: {var.dtype}")
#     print("  Encoding:", var.encoding)


In [ ]:
data_path = 'Model_Data/CompleteData/withRI'
directory_path = Path(data_path)
files_only = [data_path + '/' + entry.name for entry in directory_path.iterdir() if entry.is_file() and entry.name != '.gitkeep']
ds_cleaned = xr.open_mfdataset(files_only, concat_dim = 'frame_number', combine = 'nested')

In [ ]:
ds_cleaned = ds_cleaned.compute()
ds_cleaned = ds_cleaned.dropna(dim = 'frame_number', how = 'any')
ds_cleaned = ds_cleaned.where(ds_cleaned['center_lat'] > 0, drop = True)
ds_cleaned = ds_cleaned.where(ds_cleaned['max_wind'] > 17, drop = True)



** Start Here if CombinedDataset.nc is already created. **

In [ ]:
ds_cleaned = xr.open_dataset('CombinedDataset.nc')

In [ ]:
# Creating the values to grade against. (Minimum Central Pressure, Maximum Surface Wind, 24hr Pressure change,
#                                        24hr Wind Change, and Radius of Maximum Winds)

mslp = ds_cleaned['center_pressure'].values
max_winds = ds_cleaned['max_wind'].values
pressure_change = ds_cleaned['center_pressure_24hrs'].values - mslp
wind_change = ds_cleaned['max_wind_24hrs'].values - max_winds
category = []


wind_change_min = np.min(wind_change)
wind_change_max = np.max(wind_change)
windspeed_min = np.min(max_winds)
windspeed_max = np.max(max_winds)


for wind in max_winds:
    if  16 <= wind < 33:
        category.append(0)
    elif 33 <= wind < 43:
        category.append(1)
    elif 43 <= wind < 50:
        category.append(2)
    elif 50 <= wind < 58:
        category.append(3)
    elif 58 <= wind < 70:
        category.append(4)
    elif 70 <= wind:
        category.append(5)
# max_windspeed_radius = []
# for radius in max_windspeed_coords['radius']:
#     max_windspeed_radius.append(ds_cleaned.radius.isel(radius = radius).values.squeeze())
# max_windspeed_radius = np.squeeze(max_windspeed_radius)

# max_windspeed_radius_scaled= (max_windspeed_radius/np.max(max_windspeed_radius))*30

In [ ]:
max_winds

In [ ]:
storm_strenght_bins = [17,33,43,50,58,70, np.max(max_winds)+1]
bin_centers = [(storm_strenght_bins[i] + storm_strenght_bins[i+1]) / 2 for i in range(len(storm_strenght_bins)-1)]
bin_names = ['Tropical Storms', 'Cat 1', 'Cat 2', 'Cat 3', 'Cat 4', 'Cat 5']

plt.hist(max_winds, bins = storm_strenght_bins, rwidth = 0.75)
plt.xticks(bin_centers, bin_names)
plt.title('Distribution of Storm Strength within the Dataset')


In [ ]:
lats = ds_cleaned['center_lat'].isel(levels = 0).values
lons = ds_cleaned['center_lon'].isel(levels = 0).values


fig, ax = plt.subplots(nrows = 1, ncols = 1, figsize = (10,8), subplot_kw = {'projection': ccrs.PlateCarree()})

im = ax.scatter(lons, lats, transform = ccrs.PlateCarree(), s = 3, c = category, cmap = 'spectral')
ax.add_feature(CFeature.COASTLINE)
ax.set_title('Storm Locations')
fig.colorbar(im, location = 'bottom')

In [ ]:
# Normalizing data assuming gaussian distribution

# normalized_data_gh_sfc = utils.standard_data_normalizer(ds_cleaned['gh'].isel(levels = 0))
# normalized_data_t_sfc = utils.standard_data_normalizer(ds_cleaned['t'].isel(levels = 0))
# normalized_data_rh_sfc = utils.standard_data_normalizer(ds_cleaned['rh'].isel(levels = 0))
# normalized_data_gh_700 = utils.standard_data_normalizer(ds_cleaned['gh'].isel(levels = 1))
# normalized_data_t_700 = utils.standard_data_normalizer(ds_cleaned['t'].isel(levels = 1))
# normalized_data_rh_700 = utils.standard_data_normalizer(ds_cleaned['rh'].isel(levels = 1))
# normalized_data_gh_200 = utils.standard_data_normalizer(ds_cleaned['gh'].isel(levels = 2))
# normalized_data_t_200 = utils.standard_data_normalizer(ds_cleaned['t'].isel(levels = 2))
# normalized_data_rh_200 = utils.standard_data_normalizer(ds_cleaned['rh'].isel(levels = 2))


In [ ]:
# plt.hist(normalized_data_rh_700.reshape(-1,1))

In [ ]:
# Normalizing data while keeping underlying distribution

normalized_data_gh_sfc = utils.minmax_data_normalizer(ds_cleaned['gh'].isel(levels = 0))
normalized_data_t_sfc = utils.minmax_data_normalizer(ds_cleaned['t'].isel(levels = 0))
normalized_data_rh_sfc = utils.minmax_data_normalizer(ds_cleaned['rh'].isel(levels = 0))
normalized_data_gh_700 = utils.minmax_data_normalizer(ds_cleaned['gh'].isel(levels = 1))
normalized_data_t_700 = utils.minmax_data_normalizer(ds_cleaned['t'].isel(levels = 1))
normalized_data_rh_700 = utils.minmax_data_normalizer(ds_cleaned['rh'].isel(levels = 1))
normalized_data_gh_200 = utils.minmax_data_normalizer(ds_cleaned['gh'].isel(levels = 2))
normalized_data_t_200 = utils.minmax_data_normalizer(ds_cleaned['t'].isel(levels = 2))
normalized_data_rh_200 = utils.minmax_data_normalizer(ds_cleaned['rh'].isel(levels = 2))


In [ ]:
plt.hist(normalized_data_t_700.reshape(-1,1))

In [ ]:
# Constructing 700mb level datasets. Each array will be used in a UMAP reduction
# to test best combinations of data

combined_700 = np.stack((
                          normalized_data_gh_700, normalized_data_t_700, normalized_data_rh_700,
                          ), 
                          axis = 0).reshape(398, 3 * 720 * 501)
heights_700 = np.stack(normalized_data_gh_700,axis = 0).reshape(398, 1 * 720 * 501)
t_700 = np.stack(normalized_data_t_700,axis = 0).reshape(398, 1 * 720 * 501)
rh_700 = np.stack(normalized_data_rh_700,axis = 0).reshape(398, 1 * 720 * 501)


In [ ]:
# Constructing surface level datasets. Each array will be used in a UMAP reduction
# to test best combinations of data

combined_sfc = np.stack((
                          normalized_data_gh_sfc, normalized_data_t_sfc, normalized_data_rh_sfc,
                          ), 
                          axis = 0).reshape(398, 3 * 720 * 501)
heights_sfc = np.stack(normalized_data_gh_sfc,axis = 0).reshape(398, 1 * 720 * 501)
t_sfc = np.stack(normalized_data_t_sfc,axis = 0).reshape(398, 1 * 720 * 501)
rh_sfc = np.stack(normalized_data_rh_sfc,axis = 0).reshape(398, 1 * 720 * 501)

In [ ]:
# Constructing a composite surface/700mb level dataset. Array contains surface 
# temperature data, and surface and 700mb geopotential heights. Met values were
# selected based on best results from the individual UMAP reductions.

combined_sfc700_gph_t = np.stack((
                          normalized_data_gh_sfc, normalized_data_gh_700, normalized_data_t_sfc,
                          ), 
                          axis = 0).reshape(429, 3 * 720 * 501)

Creating Surface Level UMAP Reductions

In [ ]:
# reducer3 = umap.UMAP(n_components = 2, n_neighbors = 3, random_state = 42, init = 'pca')
# embedding_Fullsfc_n3 = reducer3.fit_transform(combined_sfc)
# embedding_GHsfc_n3 = reducer3.fit_transform(heights_sfc)
# embedding_Tsfc_n3 = reducer3.fit_transform(t_sfc)
# embedding_RHsfc_n3 = reducer3.fit_transform(rh_sfc)

reducer5 = umap.UMAP(n_components = 2, n_neighbors = 10, random_state = 12, init = 'pca')
# reducer5 = pacmap.PaCMAP(n_components = 2, n_neighbors = 30, MN_ratio = 0.5, FP_ratio = 2, random_state = 12)
embedding_Fullsfc_n5 = reducer5.fit_transform(combined_sfc)
embedding_GHsfc_n5 = reducer5.fit_transform(heights_sfc)
embedding_Tsfc_n5 = reducer5.fit_transform(t_sfc)
embedding_RHsfc_n5 = reducer5.fit_transform(rh_sfc)

# reducer15 = umap.UMAP(n_components = 2, n_neighbors = 15, random_state = 42, init = 'pca')
# embedding_Fullsfc_n15 = reducer15.fit_transform(combined_sfc)
# embedding_GHsfc_n15 = reducer15.fit_transform(heights_sfc)
# embedding_Tsfc_n15 = reducer15.fit_transform(t_sfc)
# embedding_RHsfc_n15 = reducer15.fit_transform(rh_sfc)

# reducer10 = umap.UMAP(n_components = 2, n_neighbors = 10, random_state = 42, init = 'pca')
# embedding_Fullsfc_n10 = reducer10.fit_transform(combined_sfc)
# embedding_GHsfc_n10 = reducer10.fit_transform(heights_sfc)
# embedding_Tsfc_n10 = reducer10.fit_transform(t_sfc)
# embedding_RHsfc_n10 = reducer10.fit_transform(rh_sfc)

# reducer20 = umap.UMAP(n_components = 2, n_neighbors = 20, random_state = 42, init = 'pca')
# embedding_Fullsfc_n20 = reducer20.fit_transform(combined_sfc)
# embedding_GHsfc_n20 = reducer20.fit_transform(heights_sfc)
# embedding_Tsfc_n20 = reducer20.fit_transform(t_sfc)
# embedding_RHsfc_n20 = reducer20.fit_transform(rh_sfc)


Creating 700mb Level UMAP Reductions

In [ ]:
# reducer3 = umap.UMAP(n_components = 2, n_neighbors = 3, random_state = 42, init = 'pca')
# embedding_Full700_n3 = reducer3.fit_transform(combined_700)
# embedding_GH700_n3 = reducer3.fit_transform(heights_700)
# embedding_T700_n3 = reducer3.fit_transform(t_700)
# embedding_RH700_n3 = reducer3.fit_transform(rh_700)

reducer5 = umap.UMAP(n_components = 2, n_neighbors = 4, random_state = 12, init = 'pca')
# reducer5 = pacmap.PaCMAP(n_components = 2, n_neighbors = 30, MN_ratio = 0.5, FP_ratio = 2, random_state = 12)
embedding_Full700_n5 = reducer5.fit_transform(combined_700)
embedding_GH700_n5 = reducer5.fit_transform(heights_700)
embedding_T700_n5 = reducer5.fit_transform(t_700)
embedding_RH700_n5 = reducer5.fit_transform(rh_700)

# reducer15 = umap.UMAP(n_components = 2, n_neighbors = 15, random_state = 42, init = 'pca')
# embedding_Full700_n15 = reducer15.fit_transform(combined_700)
# embedding_GH700_n15 = reducer15.fit_transform(heights_700)
# embedding_T700_n15 = reducer15.fit_transform(t_700)
# embedding_RH700_n15 = reducer15.fit_transform(rh_700)

# reducer10 = umap.UMAP(n_components = 2, n_neighbors = 10, random_state = 42, init = 'pca')
# embedding_Full700_n10 = reducer10.fit_transform(combined_700)
# embedding_GH700_n10 = reducer10.fit_transform(heights_700)
# embedding_T700_n10 = reducer10.fit_transform(t_700)
# embedding_RH700_n10 = reducer10.fit_transform(rh_700)

# reducer20 = umap.UMAP(n_components = 2, n_neighbors = 20, random_state = 42, init = 'pca')
# embedding_Full700_n20 = reducer20.fit_transform(combined_700)
# embedding_GH700_n20 = reducer20.fit_transform(heights_700)
# embedding_T700_n20 = reducer20.fit_transform(t_700)
# embedding_RH700_n20 = reducer20.fit_transform(rh_700)

Creating Composite UMAP Reductions

In [ ]:
# reducer3 = umap.UMAP(n_components = 2, n_neighbors = 3, random_state = 42, init = 'pca')
# embedding_composite_n3 = reducer3.fit_transform(combined_sfc700_gph_t)

reducer5 = umap.UMAP(n_components = 2, n_neighbors = 5, random_state = 42, init = 'pca')
embedding_composite_n5 = reducer5.fit_transform(combined_sfc700_gph_t)

# reducer15 = umap.UMAP(n_components = 2, n_neighbors = 15, random_state = 42, init = 'pca')
# embedding_composite_n15 = reducer15.fit_transform(combined_sfc700_gph_t)

# reducer10 = umap.UMAP(n_components = 2, n_neighbors = 10, random_state = 42, init = 'pca')
# embedding_composite_n10 = reducer10.fit_transform(combined_sfc700_gph_t)

# reducer20 = umap.UMAP(n_components = 2, n_neighbors = 20, random_state = 42, init = 'pca')
# embedding_composite_n20 = reducer20.fit_transform(combined_sfc700_gph_t)


Plotting UMAP compared to Minimum Central Pressure

In [ ]:
fig, ax = plt.subplots(nrows = 4, ncols = 2, figsize = (14,20))

ax[0,0].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
ax[0,0].set_ylabel('Geopotential Height')
ax[0,0].set_title('Surface Fields')

ax[1,0].scatter(embedding_Tsfc_n5[:,0], embedding_Tsfc_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
ax[1,0].set_ylabel('Temperature')

ax[2,0].scatter(embedding_RHsfc_n5[:,0], embedding_RHsfc_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
ax[2,0].set_ylabel('Relative Humidity')

mslp_im = ax[3,0].scatter(embedding_Fullsfc_n5[:,0], embedding_Fullsfc_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
ax[3,0].set_ylabel('GPH, T, RH')

# mslp_im = ax[4,0].scatter(embedding_RHsfc_n20[:,0], embedding_RHsfc_n20[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
# ax[4,0].set_ylabel('N Nearest Neighbors = 20')
fig.colorbar(mslp_im, ax = ax[3,0], location = 'bottom', label = 'Minimum Central Pressure (mb)')

ax[0,1].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
ax[0,1].set_title('700mb Fields')

ax[1,1].scatter(embedding_T700_n5[:,0], embedding_T700_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)

ax[2,1].scatter(embedding_RH700_n5[:,0], embedding_RH700_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)

ax[3,1].scatter(embedding_Full700_n5[:,0], embedding_Full700_n5[:,1], c = mslp, cmap = 'Spectral_r', s = 5)

# max_wind_im = ax[4,1].scatter(embedding_RHsfc_n20[:,0], embedding_RHsfc_n20[:,1], c = max_winds, cmap = 'Spectral', s = 5)
fig.colorbar(mslp_im, ax = ax[3,1], location = 'bottom', label = 'Minimum Central Pressure (mb)')

# fig.suptitle('Comparison of UMAP reductions on HAFS Sfc RH Model Output' \
#             '\n Data was filtered for winds over 17 m/s and Northern Hemisphere Storms.')
fig.text(0.5, 0.91, 'Comparison of UMAP reductions on HAFS Model Output as compared to Minimum Central Pressure', fontsize=16, ha='center', transform=fig.transFigure)
# Subtitle on a new line
fig.text(0.5, 0.90, "UMAP Parameters: Init = PCA, Nearest Neighbors = 4, Random State = 12, DensMap = False, Data Normalization = MinMax Scaler", fontsize=10, ha='center', transform=fig.transFigure)


Plotting UMAP Results Compare to Max Winds

In [ ]:
fig, ax = plt.subplots(nrows = 4, ncols = 2, figsize = (14,20))

ax[0,0].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[0,0].set_ylabel('Geopotential Height')
ax[0,0].set_title('Surface Fields')

ax[1,0].scatter(embedding_Tsfc_n5[:,0], embedding_Tsfc_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[1,0].set_ylabel('Temperature')

ax[2,0].scatter(embedding_RHsfc_n5[:,0], embedding_RHsfc_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[2,0].set_ylabel('Relative Humidity')

mslp_im = ax[3,0].scatter(embedding_Fullsfc_n5[:,0], embedding_Fullsfc_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[3,0].set_ylabel('GPH, T, RH')

# mslp_im = ax[4,0].scatter(embedding_RHsfc_n20[:,0], embedding_RHsfc_n20[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
# ax[4,0].set_ylabel('N Nearest Neighbors = 20')
fig.colorbar(mslp_im, ax = ax[3,0], location = 'bottom', label = 'Maximum Windspeed (m/s)')

ax[0,1].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[0,1].set_title('700mb Fields')

ax[1,1].scatter(embedding_T700_n5[:,0], embedding_T700_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)

ax[2,1].scatter(embedding_RH700_n5[:,0], embedding_RH700_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)

ax[3,1].scatter(embedding_Full700_n5[:,0], embedding_Full700_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)

# max_wind_im = ax[4,1].scatter(embedding_RHsfc_n20[:,0], embedding_RHsfc_n20[:,1], c = max_winds, cmap = 'Spectral', s = 5)
fig.colorbar(mslp_im, ax = ax[3,1], location = 'bottom', label = 'Maximum Windspeed (m/s)')

# fig.suptitle('Comparison of UMAP reductions on HAFS Sfc RH Model Output' \
#             '\n Data was filtered for winds over 17 m/s and Northern Hemisphere Storms.')
fig.text(0.5, 0.91, 'Comparison of UMAP reductions on HAFS Model Output as compared to Maximum Windspeed', fontsize=16, ha='center', transform=fig.transFigure)
# Subtitle on a new line
fig.text(0.5, 0.90, "UMAP Parameters: Init = PCA, Nearest Neighbors = 4, Random State = 12, DensMap = False, Data Normalization = MinMax Scaler", fontsize=10, ha='center', transform=fig.transFigure)


Plotting UMAP Results Compare to Pressure Change 24hrs

In [ ]:
fig, ax = plt.subplots(nrows = 4, ncols = 2, figsize = (12,18))

ax[0,0].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)
ax[0,0].set_ylabel('Geopotential Height')
ax[0,0].set_title('Surface')

ax[1,0].scatter(embedding_Tsfc_n5[:,0], embedding_Tsfc_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)
ax[1,0].set_ylabel('Temperature')

ax[2,0].scatter(embedding_RHsfc_n5[:,0], embedding_RHsfc_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)
ax[2,0].set_ylabel('Relative Humidity')

mslp_im = ax[3,0].scatter(embedding_Fullsfc_n5[:,0], embedding_Fullsfc_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)
ax[3,0].set_ylabel('GPH, T, RH')

# mslp_im = ax[4,0].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
# ax[4,0].set_ylabel('N Nearest Neighbors = 20')
fig.colorbar(mslp_im, ax = ax[3,0], location = 'bottom', label = '24 Hour Pressure Change (mb)')

ax[0,1].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)
ax[0,1].set_title('700mb')

ax[1,1].scatter(embedding_T700_n5[:,0], embedding_T700_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)

ax[2,1].scatter(embedding_RH700_n5[:,0], embedding_RH700_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)

max_wind_im = ax[3,1].scatter(embedding_Full700_n5[:,0], embedding_Full700_n5[:,1], c = pressure_change, cmap = 'RdBu_r', s = 5)

# max_wind_im = ax[4,1].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = max_winds, cmap = 'Spectral', s = 5)
fig.colorbar(max_wind_im, ax = ax[3,1], location = 'bottom', label = '24 Hour Pressure Change (mb)')

# fig.suptitle('Comparison of UMAP reductions on HAFS 700 GPH Model Output' \
#             '\n Data was filtered for winds over 17 m/s and Northern Hemisphere Storms.')

fig.text(0.5, 0.91, 'Comparison of UMAP reductions on HAFS Model Output as compared to 24 Hour Pressure Change', fontsize=18, ha='center', transform=fig.transFigure)
# Subtitle on a new line
fig.text(0.5, 0.90, "UMAP Parameters: Init = PCA, Nearest Neighbors = 4, Random State = 12, DensMap = False, Data Normalization = MinMax Scaler", fontsize=10, ha='center', transform=fig.transFigure)


In [ ]:
fig, ax = plt.subplots(nrows = 4, ncols = 2, figsize = (14,20))

ax[0,0].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[0,0].set_ylabel('Geopotential Height')
ax[0,0].set_title('Surface')

ax[1,0].scatter(embedding_Tsfc_n5[:,0], embedding_Tsfc_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[1,0].set_ylabel('Temperature')

ax[2,0].scatter(embedding_RHsfc_n5[:,0], embedding_RHsfc_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[2,0].set_ylabel('Relative Humidity')

mslp_im = ax[3,0].scatter(embedding_Fullsfc_n5[:,0], embedding_Fullsfc_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[3,0].set_ylabel('GPH, T, RH')

# mslp_im = ax[4,0].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
# ax[4,0].set_ylabel('N Nearest Neighbors = 20')
fig.colorbar(mslp_im, ax = ax[3,0], location = 'bottom', label = '24 Hour Maximum Windspeed Change (m/s)')

ax[0,1].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[0,1].set_title('700mb')

ax[1,1].scatter(embedding_T700_n5[:,0], embedding_T700_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)

ax[2,1].scatter(embedding_RH700_n5[:,0], embedding_RH700_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)

max_wind_im = ax[3,1].scatter(embedding_Full700_n5[:,0], embedding_Full700_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)

# max_wind_im = ax[4,1].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = max_winds, cmap = 'Spectral', s = 5)
fig.colorbar(max_wind_im, ax = ax[3,1], location = 'bottom', label = '24 Hour Maximum Windspeed Change (m/s)')

# fig.suptitle('Comparison of UMAP reductions on HAFS 700 GPH Model Output' \
#             '\n Data was filtered for winds over 17 m/s and Northern Hemisphere Storms.')

fig.text(0.5, 0.91, 'Comparison of UMAP reductions on HAFS Model Output as compared to 24 Hour Maximum Windspeed Change', fontsize=18, ha='center', transform=fig.transFigure)
# Subtitle on a new line
fig.text(0.5, 0.90, "UMAP Parameters: Init = PCA, Nearest Neighbors = 4, Random State = 12, DensMap = False, Data Normalization = MinMax Scaler", fontsize=10, ha='center', transform=fig.transFigure)


In [ ]:
fig, ax = plt.subplots(nrows = 2, ncols = 2, figsize = (14,14))

sfc_ws_im = ax[0,0].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[0,0].set_ylabel('Surface')
ax[0,0].set_title('Max Windspeed Shading')

ax[1,0].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = max_winds, cmap = 'Spectral', s = 5)
ax[1,0].set_ylabel('700mb')



# mslp_im = ax[4,0].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = mslp, cmap = 'Spectral_r', s = 5)
# ax[4,0].set_ylabel('N Nearest Neighbors = 20')
fig.colorbar(sfc_ws_im, ax = ax[1,0], location = 'bottom', label = '24 Hour Maximum Windspeed Change (m/s)')

ax[0,1].scatter(embedding_GHsfc_n5[:,0], embedding_GHsfc_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)
ax[0,1].set_title('Max Windspeed Change Shading')

sfc_ws_change_im = ax[1,1].scatter(embedding_GH700_n5[:,0], embedding_GH700_n5[:,1], c = wind_change, cmap = 'RdBu', s = 5)

# max_wind_im = ax[4,1].scatter(embedding_GH700_n20[:,0], embedding_GH700_n20[:,1], c = max_winds, cmap = 'Spectral', s = 5)
fig.colorbar(sfc_ws_change_im, ax = ax[1,1], location = 'bottom', label = '24 Hour Maximum Windspeed Change (m/s)')

# fig.suptitle('Comparison of UMAP reductions on HAFS 700 GPH Model Output' \
#             '\n Data was filtered for winds over 17 m/s and Northern Hemisphere Storms.')

fig.suptitle( 'Comparison of UMAP reductions on HAFS Model Output as compared to 24 Hour Maximum Windspeed Change', fontsize=18, ha='center', transform=fig.transFigure)
# Subtitle on a new line
fig.text(0.5, 0.95, "UMAP Parameters: Init = PCA, Nearest Neighbors = 4, Random State = 12, DensMap = False, Data Normalization = MinMax Scaler", fontsize=10, ha='center', transform=fig.transFigure)


Computing Composites for two strengthening and two weakening clusters in the 700mb GPH

In [ ]:

filter_y = (embedding_GH700_n5[:,1] > 3) & (embedding_GH700_n5[:,1] < 4)
filter_x = (embedding_GH700_n5[:,0] > -5) & (embedding_GH700_n5[:,0] < -3)
cluster1_filter = filter_x * filter_y

filter_y = (embedding_GH700_n5[:,1] > 2) & (embedding_GH700_n5[:,1] < 4)
filter_x = (embedding_GH700_n5[:,0] > 7) & (embedding_GH700_n5[:,0] < 10)
cluster2_filter = filter_x * filter_y

filter_y = (embedding_GH700_n5[:,1] > 1) & (embedding_GH700_n5[:,1] < 2)
filter_x = (embedding_GH700_n5[:,0] > -1) & (embedding_GH700_n5[:,0] < 0.2)
cluster3_filter = filter_x * filter_y

filter_y = (embedding_GH700_n5[:,1] > 5.2) & (embedding_GH700_n5[:,1] < 6.2)
filter_x = (embedding_GH700_n5[:,0] > 0) & (embedding_GH700_n5[:,0] < 2)
cluster4_filter = filter_x * filter_y

In [ ]:
UMAP_embedding = embedding_GH700_n5
composite_filter = cluster4_filter
cluster_wind_change = wind_change[composite_filter]
plt.scatter(UMAP_embedding[:,0], UMAP_embedding[:,1],color = 'black', s = 5, alpha = 0.05)
plt.scatter(UMAP_embedding[composite_filter,0], UMAP_embedding[composite_filter,1], 
            cmap = 'Spectral', c = cluster_wind_change, s = 5,
            vmin = windspeed_min,
            vmax = windspeed_max)

In [ ]:
cluster1_wind_change = wind_change[cluster1_filter]
avg_cluster1_wind_change = np.nanmean(cluster1_wind_change)
cluster1_winds = max_winds[cluster1_filter]
avg_cluster1_wind = np.nanmean(cluster1_winds)
med_cluster1_wind = np.nanmedian(cluster1_winds)
cluster1_size = len(cluster1_winds)

cluster2_wind_change = wind_change[cluster2_filter]
avg_cluster2_wind_change = np.nanmean(cluster2_wind_change)
cluster2_winds = max_winds[cluster2_filter]
avg_cluster2_wind = np.nanmean(cluster2_winds)
med_cluster2_wind = np.nanmedian(cluster2_winds)
cluster2_size = len(cluster2_winds)

cluster3_wind_change = wind_change[cluster3_filter]
avg_cluster3_wind_change = np.nanmean(cluster3_wind_change)
cluster3_winds = max_winds[cluster3_filter]
avg_cluster3_wind = np.nanmean(cluster3_winds)
med_cluster3_wind = np.nanmedian(cluster3_winds)
cluster3_size = len(cluster3_winds)

cluster4_wind_change = wind_change[cluster4_filter]
avg_cluster4_wind_change = np.nanmean(cluster4_wind_change)
cluster4_winds = max_winds[cluster4_filter]
avg_cluster4_wind = np.nanmean(cluster4_winds)
med_cluster4_wind = np.nanmedian(cluster4_winds)
cluster4_size = len(cluster4_winds)

wind_change_min = np.min(wind_change)
wind_change_max = np.max(wind_change)
windspeed_min = np.min(max_winds)
windspeed_max = np.max(max_winds)

In [ ]:
cluster4_size

In [ ]:
med_cluster1_wind

In [ ]:
def get_CI(field_anomalies, filter):
    n_samples = 1000

    cluster_size = np.sum(filter)
    cluster_mean = np.mean(field_anomalies[filter,:,:])
    bootstrap_anomalies = []
    for n in range(n_samples):
            indices = np.random.choice(field_anomalies.shape[0], size = cluster_size, replace = True)
            sample_fields = field_anomalies[indices, :, :]
            mean_sample = np.mean(sample_fields, axis = 0)
            bootstrap_anomalies.append(cluster_mean - mean_sample)
            
 
    
    ci_lo = np.quantile(bootstrap_anomalies, 0.025, axis = 0)
    ci_hi = np.quantile(bootstrap_anomalies, 0.975, axis = 0)
    p_value = np.mean(np.array(bootstrap_anomalies) <= 0)
        

    return (ci_lo,ci_hi), p_value  
    # significantU = np.zeros([field_anomalies.shape[1], field_anomalies.shape[2]])
    # significantL = np.zeros([field_anomalies.shape[1], field_anomalies.shape[2]])
    # significantU[cluster_field_mean_anomaly > upper95] = True
    # significantL[cluster_field_mean_anomaly < lower05] = True

    # significant_areas = significantL + significantU
    # significant_areas_coords = np.where(significant_areas == 1)
    # sig_coords_radius = []

In [ ]:
def difference_of_means(data1, data2, axis):
    return np.mean(data1, axis = axis) - np.mean(data2, axis = axis)

level = 1
composite_filter = cluster4_filter

angle = ds_cleaned.angle.values
radius = ds_cleaned.radius.values

ds_gph = ds_cleaned['gh'].isel(levels = level).values
ds_t = ds_cleaned['t'].isel(levels = level).values
max_winds = ds_cleaned['max_wind'].values[:,0]
wind_change = ds_cleaned['24HrMaxWinds'].isel(levels = 0).values - max_winds

cluster_wind_change = wind_change[composite_filter]
avg_cluster_wind_change = np.nanmean(cluster_wind_change)
cluster_winds = max_winds[composite_filter]
avg_cluster_wind = np.nanmean(cluster_winds)
cluster_size =np.sum(composite_filter)
#### GH Composite ####
mean_field_gh = np.mean(ds_gph, axis = 0)
field_anomalies_gh = ds_gph - mean_field_gh
cluster_field_anomalies_gh = field_anomalies_gh[composite_filter, :, :]
cluster_field_mean_anom_gh = np.mean(cluster_field_anomalies_gh, axis = 0)
cluster_field_mean_gh = np.mean(ds_gph[composite_filter,:,:], axis = 0)

CI_gh, pvalue_gh = get_CI(field_anomalies_gh, composite_filter)
            
sig_gh_x, sig_gh_y = np.where(np.logical_or((CI_gh[0] > 0), ( CI_gh[1] < 0)))
sig_gh_x = sig_gh_x/2
sig_gh_y = sig_gh_y/2
# sig_upper_gh = np.where(CI_gh_hi < np.quantile(cluster_field_anomalies_gh, .025, axis = 0))
# sig_upper_x_gh = sig_upper_gh[0]/2
# sig_upper_y_gh = sig_upper_gh[1]/2

# sig_lower_gh = np.where(CI_gh_lo > np.quantile(cluster_field_anomalies_gh, .975, axis = 0))
# sig_lower_x_gh = sig_lower_gh[0]/2
# sig_lower_y_gh = sig_lower_gh[1]/2

#### T Composite #####
mean_field_t = np.mean(ds_t, axis = 0)
field_anomalies_t = ds_t - mean_field_t
cluster_field_anomalies_t = field_anomalies_t[composite_filter, :, :]
cluster_field_mean_anom_t = np.mean(cluster_field_anomalies_t, axis = 0)
cluster_field_mean_t = np.mean(ds_t[composite_filter,:,:], axis = 0)

CI_t, pvalue_t = get_CI(field_anomalies_t, composite_filter)

# sig_upper_t = np.where(CI_t_hi < np.quantile(cluster_field_anomalies_t, .025, axis = 0))
# sig_upper_x_t = sig_upper_t[0]/2
# sig_upper_y_t = sig_upper_t[1]/2

# sig_lower_t = np.where(CI_t_lo > np.quantile(cluster_field_anomalies_t, .975, axis = 0))
# sig_lower_x_t = sig_lower_t[0]/2
# sig_lower_y_t = sig_lower_t[1]/2

sig_t_x, sig_t_y = np.where(np.logical_or((CI_t[0] > 0), ( CI_t[1] < 0)))
sig_t_x = sig_t_x/2
sig_t_y = sig_t_y/2

# fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize=(12, 12), subplot_kw= {'projection':'polar'})
# Theta, R = np.meshgrid(angle, radius)
# height_im = ax[0].pcolormesh(angle, radius, cluster_field_anomalies_gh, cmap = 'RdBu', shading = 'nearest')
# ax[0].scatter(sig_upper_x_gh, sig_upper_y_gh, alpha = 0.01, color = 'black', marker = 'x', s = 1)
# ax[0].grid(False)
# ax[0].set_theta_direction(-1)
# ax[0].set_theta_offset(np.pi/2)
    # ax[0].text(5.48,450, f'24hr Mean Max Wind Change: {avg_cluster_wind_change:.2f} m/s')
    # ax[0].set_title('Geopotential Height')
    # ax00_inset = ax[0].inset_axes([.8, .65, .5, .4])
    # ax00_inset.scatter(UMAP_embedding[:,0], UMAP_embedding[:,1],color = 'black', s = 5, alpha = 0.05)
    # ax00_inset.scatter(UMAP_embedding[filter,0], UMAP_embedding[filter,1], 
    #                c = cluster_winds, cmap = 'Spectral', s = 5,
    #                vmin = windspeed_min,
    #                vmax = windspeed_max)
    # fig.colorbar(height_im,cax = ax[0], location = 'bottom')

In [ ]:
UMAP_embedding = embedding_GH700_n5

cluster_name = 'Cluster 4'
fig, ax = plt.subplots(nrows = 2, ncols = 2, figsize=(12, 12), subplot_kw= {'projection':'polar'})
Theta, R = np.meshgrid(angle, radius)
height_im = ax[0,0].contourf(angle, radius, cluster_field_mean_gh.T, cmap = 'RdBu_r')
# ax[0,0].scatter(sig_upper_x_gh[::50], sig_upper_y_gh[::50], alpha = 0.1, color = 'black', marker = 'x', s = 10)
# ax[0,0].scatter(sig_lower_x_gh[::50], sig_lower_y_gh[::50], alpha = 0.1, color = 'black', marker = 'x', s = 10)
ax[0,0].grid(False)
ax[0,0].set_theta_direction(-1)
ax[0,0].set_theta_offset(np.pi/2)
ax[0,0].set_title('Cluster Geopotential Height')
fig.colorbar(height_im, ax = ax[0,0], location = 'bottom', label = 'Geopotential Height (m)')

temp_im = ax[0,1].contourf(angle, radius, cluster_field_mean_t.T, cmap = 'viridis')
# ax[0,1].scatter(sig_upper_x_t[::60], sig_upper_y_t[::60], alpha = 0.15, color = 'black', marker = '*', s = 5)
# ax[0,1].scatter(sig_lower_x_t[::60], sig_lower_y_t[::60], alpha = 0.15, color = 'black', marker = '*', s = 5)
ax[0,1].grid(False)
ax[0,1].set_theta_direction(-1)
ax[0,1].set_theta_offset(np.pi/2)
ax[0,1].set_title('Cluster Temperature')
fig.colorbar(temp_im, ax = ax[0,1], location = 'bottom', label = 'Temperature (C)')

ax00_inset = ax[0,0].inset_axes([1.15, .66, .5, .4])
ax00_inset.scatter(UMAP_embedding[:,0], UMAP_embedding[:,1],color = 'black', s = 5, alpha = 0.05)
ax00_inset.scatter(UMAP_embedding[composite_filter,0], UMAP_embedding[composite_filter,1], 
                   c = cluster_winds, cmap = 'Spectral', s = 5,
                   vmin = windspeed_min,
                   vmax = windspeed_max)


height_anom_im = ax[1,0].contourf(angle, radius, cluster_field_mean_anom_gh.T, cmap = 'RdBu_r')
# ax[1,0].scatter(sig_gh_x[::30], sig_gh_y[::30], alpha = 0.1, color = 'black', marker = 'x', s = 10)
# ax[1,0].scatter(sig_lower_x_gh[::40], sig_lower_y_gh[::40], alpha = 0.1, color = 'black', marker = 'x', s = 10)
ax[1,0].grid(False)
ax[1,0].set_theta_direction(-1)
ax[1,0].set_theta_offset(np.pi/2)
ax[1,0].set_title('Cluster Geopotential Height Anomaly')
fig.colorbar(height_anom_im, ax = ax[1,0], location = 'bottom', label = 'Geopotential Height Anomaly (m)')

temp_anom_im = ax[1,1].contourf(angle, radius, cluster_field_mean_anom_t.T, cmap = 'viridis')
# ax[1,1].scatter(sig_t_x[::20], sig_t_y[::20], alpha = 0.05, color = 'black', marker = '*', s = 1)
# ax[1,1].scatter(sig_lower_x_t, sig_lower_y_t, alpha = 0.05, color = 'black', marker = '*', s = 1)
ax[1,1].grid(False)
ax[1,1].set_theta_direction(-1)
ax[1,1].set_theta_offset(np.pi/2)
ax[1,1].set_title('Cluster Temperature Anomaly')
fig.colorbar(temp_anom_im, ax = ax[1,1], location = 'bottom', label = 'Temperature Anomaly (C)')

fig.suptitle(f'{cluster_name} \nCluster Average Windspeed: {avg_cluster_wind:.2f} m/s \nCluster Average 24hr Storm Intensity Change: {avg_cluster_wind_change:.2f} m/s \n Cluster Size: {cluster_size}')

In [ ]:
lats1 = ds_cleaned['center_lat'].isel(levels = 0).values[cluster1_filter]
lons1 = ds_cleaned['center_lon'].isel(levels = 0).values[cluster1_filter]
lats2 = ds_cleaned['center_lat'].isel(levels = 0).values[cluster2_filter]
lons2 = ds_cleaned['center_lon'].isel(levels = 0).values[cluster2_filter]
lats3 = ds_cleaned['center_lat'].isel(levels = 0).values[cluster3_filter]
lons3 = ds_cleaned['center_lon'].isel(levels = 0).values[cluster3_filter]
lats4 = ds_cleaned['center_lat'].isel(levels = 0).values[cluster4_filter]
lons4 = ds_cleaned['center_lon'].isel(levels = 0).values[cluster4_filter]


fig, ax = plt.subplots(nrows = 4, ncols = 1, figsize = (10,8), subplot_kw = {'projection': ccrs.PlateCarree()})

ax[0].scatter(lons1, lats1, transform = ccrs.PlateCarree(), s = 3)
ax[0].add_feature(CFeature.COASTLINE)

ax[0].set_title('Cluster 1')

ax[1].scatter(lons2, lats2, transform = ccrs.PlateCarree(), s = 3)
ax[1].add_feature(CFeature.COASTLINE)
ax[1].set_title('Cluster 2')

ax[2].scatter(lons3, lats3, transform = ccrs.PlateCarree(), s = 3)
ax[2].add_feature(CFeature.COASTLINE)
ax[2].set_title('Cluster 3')

ax[3].scatter(lons4, lats4, transform = ccrs.PlateCarree(), s = 3)
ax[3].add_feature(CFeature.COASTLINE)
ax[3].set_title('Cluster 4')

fig.suptitle('Storm Locations by Cluster')


Computing composites for both clusters the 700mb GPH, T, RH combined cluster

In [ ]:
filter_y = embedding_Full700_n5[:,1] > 5
filter_x = embedding_Full700_n5[:,0] < 0
cluster1_filter_700mbFull = filter_x * filter_y

filter_y = embedding_Full700_n5[:,1] < 6
filter_x = embedding_Full700_n5[:,0] > 10
cluster2_filter_700mbFull = filter_x * filter_y

cluster1_wind_change = wind_change[cluster1_filter_700mbFull]
cluster2_wind_change = wind_change[cluster2_filter_700mbFull]

